# LegacyBridge — Free-tier Colab (T4) model server for CloudCode

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Copy the URL printed by step 5 into CloudCode's `orchestrator_config.json` (`tunnel_url`) and set `"model": "qwen2.5-coder:14b"`.

What changed from the previous notebook, and why:

| Before | Now |
|---|---|
| `Qwen2.5-Coder-3B` (fp16), but reported as `qwen2.5-coder:32b` | **`qwen2.5-coder:14b`** via Ollama (Q4_K_M, ~9 GB): the strongest Qwen coder that fits a free T4 with a 16K context. The real model name is reported. |
| Only the **last user message** reached the model | The **whole conversation** (system + history) is forwarded |
| `max_new_tokens=1000` hard-coded; `max_tokens`/`temperature` ignored | Requested `max_tokens` and `temperature` are honoured (capped at `MAX_OUTPUT_TOKENS`) |
| Always `finish_reason: "stop"`; word counts as tokens | True `finish_reason` (`length` when cut off) and real token counts |
| Replies over ~100 s were dropped by Cloudflare (524) | The bridge sends a space every 10 s while generating (valid leading JSON whitespace), so long replies survive |

Keep this tab open; run the last cell to keep the session alive.

In [ ]:
# 1. Check the GPU (should say Tesla T4, ~15 GB)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# 2. Install Ollama and the bridge's dependencies
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q fastapi uvicorn
!ollama --version

In [ ]:
# 3. Start the Ollama server with a context window large enough for CloudCode's prompts
import json, os, subprocess, time, urllib.request

MODEL = "qwen2.5-coder:14b"   # fits a free T4. Faster but weaker: "qwen2.5-coder:7b"
CONTEXT = 16384               # tokens of prompt + reply; Ollama's small default silently drops the prompt start
OLLAMA = "http://127.0.0.1:11434"

def ollama_up():
    try:
        return urllib.request.urlopen(OLLAMA + "/api/tags", timeout=3).status == 200
    except Exception:
        return False

if not ollama_up():
    env = os.environ.copy()
    env.update({
        "OLLAMA_HOST": "127.0.0.1:11434",
        "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
        "OLLAMA_KEEP_ALIVE": "-1",          # keep the model loaded between requests
        "OLLAMA_NUM_PARALLEL": "1",         # one request at a time: all GPU memory for one context
        "OLLAMA_FLASH_ATTENTION": "1",
        "OLLAMA_KV_CACHE_TYPE": "q8_0",     # halves context memory so 16K fits beside the 14B weights
    })
    ollama_log = open("/content/ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("Ollama running:", ollama_up())

In [ ]:
# 4. Download the model (~9 GB, a few minutes) and check it runs fully on the GPU
!ollama pull {MODEL}

req = urllib.request.Request(OLLAMA + "/api/generate", data=json.dumps({
    "model": MODEL, "prompt": "Write a Python function that checks whether a string is a palindrome.",
    "stream": False, "options": {"num_predict": 200}}).encode(), headers={"Content-Type": "application/json"})
result = json.loads(urllib.request.urlopen(req, timeout=900).read())
TOKENS_PER_SECOND = result["eval_count"] / (result["eval_duration"] / 1e9)
print(f"Generation speed: {TOKENS_PER_SECOND:.1f} tokens/s")

ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
if "100% GPU" not in ps:
    print("WARNING: the model is not fully on the GPU (check the PROCESSOR column). "
          "It will be very slow. Restart the runtime, or switch MODEL to qwen2.5-coder:7b.")

# Longest reply the bridge will produce; CloudCode continues longer files automatically.
MAX_OUTPUT_TOKENS = 4096
print(f"A full {MAX_OUTPUT_TOKENS}-token reply takes about {MAX_OUTPUT_TOKENS / TOKENS_PER_SECOND / 60:.1f} min at this speed.")

In [ ]:
# 5a. OpenAI-compatible bridge in front of Ollama (full history, honest limits, Cloudflare-safe)
import asyncio, json, threading, time, urllib.request
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse

def model_meta():
    return {"id": MODEL, "object": "model", "owned_by": "ollama", "backend": "ollama",
            "context_window": CONTEXT, "max_output_tokens": MAX_OUTPUT_TOKENS,
            "supports_full_history": True, "usage_source": "tokenizer"}

def forward(body):
    """Send one chat completion to Ollama and return its OpenAI-format JSON."""
    req = urllib.request.Request(OLLAMA + "/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=3600) as res:
        return json.loads(res.read())

def prepare(body):
    """Normalise a client request: loaded model, non-streaming, capped output budget."""
    messages = body.get("messages")
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages must be a non-empty list")
    requested = body.get("max_tokens") or MAX_OUTPUT_TOKENS
    out = {"model": MODEL, "messages": messages, "stream": False,
           "max_tokens": max(1, min(int(requested), MAX_OUTPUT_TOKENS)),
           "temperature": float(body.get("temperature", 0.2))}
    for key in ("top_p", "stop", "seed"):
        if key in body:
            out[key] = body[key]
    return out

app = FastAPI(title="LegacyBridge (Ollama)")

@app.get("/")
@app.get("/models")
@app.get("/v1/models")
@app.get("/api/tags")
def models():
    meta = model_meta()
    return {"object": "list", "status": "healthy", "data": [meta], "models": [meta]}

@app.post("/v1/chat/completions")
@app.post("/chat/completions")
async def chat(request: Request):
    body = prepare(await request.json())
    prompt_chars = sum(len(str(m.get("content", ""))) for m in body["messages"])
    print(f"\n>>> request: {len(body['messages'])} messages, {prompt_chars} chars, max_tokens={body['max_tokens']}")

    async def respond():
        # Cloudflare drops a request whose first byte takes over 100 s. Send
        # whitespace (valid before a JSON document) until the reply is ready.
        loop = asyncio.get_running_loop()
        started = time.time()
        job = loop.run_in_executor(None, forward, body)
        yield b" "
        while True:
            done, _ = await asyncio.wait({job}, timeout=10)
            if done:
                break
            yield b" "
        try:
            reply = job.result()
            reply["model"] = MODEL
            choice, usage = reply["choices"][0], reply.get("usage", {})
            took = time.time() - started
            print(f"<<< reply: finish={choice.get('finish_reason')} tokens={usage.get('completion_tokens')} "
                  f"in {took:.0f}s ({(usage.get('completion_tokens') or 0) / max(took, 1e-6):.1f} tok/s)")
        except Exception as exc:
            print("!!! bridge error:", repr(exc))
            reply = {"error": {"message": str(exc), "type": "bridge_error"}}
        yield json.dumps(reply).encode()

    return StreamingResponse(respond(), media_type="application/json")


In [ ]:
# 5b. Start the bridge and a Cloudflare tunnel, then print the public URL
import os, re, subprocess, threading, time, urllib.request
import uvicorn

if not globals().get("BRIDGE_STARTED"):
    threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                     daemon=True).start()
    BRIDGE_STARTED = True
    time.sleep(2)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
                   shell=True, check=True)

public_url = globals().get("public_url")
if not public_url:
    cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in cf_proc.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    # Keep draining cloudflared's log so its pipe never fills and stalls the tunnel.
    threading.Thread(target=lambda: [None for _ in cf_proc.stderr], daemon=True).start()

time.sleep(5)
try:
    print("Public health check:", urllib.request.urlopen(public_url + "/v1/models", timeout=30).read().decode()[:200])
except Exception as exc:
    print("Public health check not ready yet (DNS can take ~30 s):", exc)

print("\n" + "=" * 70)
print(f"  COLAB PUBLIC URL : {public_url}")
print(f"  MODEL            : {MODEL}   (set this as \"model\" in orchestrator_config.json)")
print("=" * 70)

In [ ]:
# 6. Keep the session alive and show status every 5 minutes (stop with the square button)
import time, urllib.request
while True:
    try:
        ok = urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=10).status == 200
    except Exception:
        ok = False
    print(time.strftime("%H:%M:%S"), "bridge OK" if ok else "bridge NOT responding", "|", public_url)
    time.sleep(300)